# Phase 1: Data Preparation & Cleaning (Python / Pandas)
**DRC Measles Surveillance Capstone**

Steps:
1. Load and explore the data
2. Clean the date columns (placeholders, mixed formats, typos), with an audit log
3. Check the logical order of the anchor dates
4. Standardise the age into **Age in Years**
5. Standardise **Vaccination Status** and **Vaccine Doses**
6. Tidy the other columns used later
7. Apply the same cleaning to the test data and save



In [1]:
import pandas as pd
import numpy as np

## 1. Load and explore the data

In [2]:
train = pd.read_csv("train_data.csv")
test = pd.read_csv("test_data.csv")

display(train.head(), train.shape, test.shape)

,Reporting Date,Patient ID,Epi week,Zone (Kinshasa ONLY),Reporting Health Facility,Reporting District,Reporting County,Disease/Condition,Community Case Detection,Patient Sex,...,Final Laboratory Result,Date of Result Emailed to CHT,Time of Result Emailed to CHT,Lab Specimen ID,Laboratory Comment,Final Epi-classification,Rubella Date Tested,Rubella Results,Critical Surveillance Issues,Unnamed: 60
0,2026-03-18 00:00:00,MON-LRUG08-025,12,1100A-2,Hôpital Biamba Marie Mutombo,Lubumbashi,Kinshasa,Measles,No,Male,...,Suspected,No specimen tested,No specimen tested,No lab result,No comment,Clinically compatible,Not provided,Not provided,Not provided,NaN
1,2026-03-27 00:00:00,BON-LRG4N6-011,13,Hors Zone,Kelebei Clinique,Buta,Kongo Central,Measles,No,Male,...,Not applicable,Not applicable,Not applicable,Not applicable,Not applicable,Clinically compatible,Not applicable,Not applicable,NaN,NaN
2,2026-04-13 00:00:00,BON-LRJLM1-475,16,Hors Zone,Centre Pédiatrique de Kimbondo,Lubero,Kongo Central,Measles,No,Male,...,Suspected,No specimen collected,No specimen collected,NaN,Not applicable,Clinically compatible,Not applicable,Not applicable,Community transmission,No
3,2026-02-24 00:00:00,GRB-LRQ6T6-017,9,Hors Zone,Well Baby Clinique,Bandalungwa,Kwilu,Measles,No,Male,...,Pending,Pending,Pending,Pending,Pending,Suspected,Pending,Pending,Not available,NaN
4,2026-04-09 00:00:00,BOM-LRT-6S5-023,15,Hors Zone,Suehn clinic,Aru,Tshopo,Measles,Yes,Male,...,Not applicable,No specimen collected,No specimen collected,NaN,Not applicable,Clinically compatible,Not applicable,Not applicable,Patient was seen and treated at the clinic and...,No


(4107, 61)

(1027, 61)

In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 4107 entries, 0 to 4106
Data columns (total 61 columns):
 #   Column                                 Non-Null Count  Dtype
---  ------                                 --------------  -----
 0   Reporting Date                         4107 non-null   str  
 1   Patient ID                             4019 non-null   str  
 2   Epi week                               4107 non-null   int64
 3   Zone (Kinshasa ONLY)                   3696 non-null   str  
 4   Reporting Health Facility              4107 non-null   str  
 5   Reporting District                     4107 non-null   str  
 6   Reporting County                       4107 non-null   str  
 7   Disease/Condition                      4107 non-null   str  
 8   Community Case Detection               4104 non-null   str  
 9   Patient Sex                            4106 non-null   str  
 10  Patient Age                            4106 non-null   str  
 11  Age Type                               41

In [4]:
# number of empty cells in each column
train.isna().sum()

Reporting Date                     0
Patient ID                        88
Epi week                           0
Zone (Kinshasa ONLY)             411
Reporting Health Facility          0
                                ... 
Final Epi-classification          13
Rubella Date Tested               81
Rubella Results                   82
Critical Surveillance Issues    1044
Unnamed: 60                     3670
Length: 61, dtype: int64

## 2. Cleaning the date columns

Let's look at what is actually inside a date column. Besides real dates there are text placeholders
(`No specimen collected`, `Pending`, `Not done` ...) and messy formats such as `30_dec-25`, `27-Feb.26` or `8-Ffeb-26`.

In [5]:
date_columns = [
    "Reporting Date",
    "Date of Birth",
    "Date of Onset/Emergence",
    "Date of Detection/Seen",
    "Date Specimen Collected",
    "Date Specimen Sent",
    "Date Specimen Received at Lab",
    "Date Lab Tested/ Reviewed",
]

train["Date Specimen Collected"].value_counts().head(15)

Date Specimen Collected
No specimen collected      2189
No Specimen collected       169
Not applicable              122
No specimen collected        69
Not done                     49
No specimen  collected       48
No specimen collected        47
2026-02-04 00:00:00          40
2026-01-19 00:00:00          37
2026-02-03 00:00:00          37
2026-01-21 00:00:00          32
2026-02-02 00:00:00          32
No Specimen Collected        27
No sample collected          25
2026-03-04 00:00:00          24
Name: count, dtype: int64

### 2.1 A function to clean the date text
* If the cell has **no digits at all**, it is a placeholder (`Pending`, `Not done`...) → `NaN`
* If it is a **time** typed in a date column (`11:30am`) → `NaN`
* Otherwise fix the separators (`_ . ; /` → `-`), the misspelled months (`Ffeb` → `Feb`) and 3-digit years (`-026` → `-26`)

In [6]:
replacements = {
    " 00:00:00": "",                                   # remove the time part
    "_": "-", ".": "-", ";": "-", "/": "-", " ": "-",   # separators
    "=": "", ":": "", "!": "1", "--": "-",
    # misspelled month names found in the data
    "Ffeb": "Feb", "Fab": "Feb", "Mare": "Mar", "-Mat-": "-Mar-", "Lan": "Jan",
    "-an-": "-Jan-", "-Jn-": "-Jan-", "Aprial": "Apr", "Appr": "Apr", "Sept": "Sep",
    "3o-": "30-", "-Jan-Jan": "-Jan-26",
    # 3-digit years
    "-026": "-26", "-206": "-26", "-226": "-26", "-025": "-25", "-024": "-24", "-023": "-23",
    "-022": "-22", "-021": "-21", "-020": "-20", "-019": "-19", "-018": "-18", "-017": "-17",
    "-016": "-16", "-015": "-15", "-014": "-14", "-013": "-13",
}

def clean_date_text(text):
    if pd.isna(text):
        return np.nan
    text = str(text).strip()
    has_digit = any(character.isdigit() for character in text)
    if not has_digit:                                   # placeholder such as "Pending"
        return np.nan
    if "am" in text.lower() or "pm" in text.lower():    # a time, not a date
        return np.nan
    for old, new in replacements.items():
        text = text.replace(old, new)
    if text.endswith("-"):                              # "27-Feb-" -> year missing, assume 2026
        text = text + "26"
    return text

# test the function on a few examples
for example in ["30_dec-25", "27-Feb.26", "8-Ffeb-26", "Pending", "2026-02-06 00:00:00", "11:30am"]:
    print(example, "->", clean_date_text(example))

30_dec-25 -> 30-dec-25
27-Feb.26 -> 27-Feb-26
8-Ffeb-26 -> 8-Feb-26
Pending -> nan
2026-02-06 00:00:00 -> 2026-02-06
11:30am -> nan


### 2.2 Convert the cleaned text into real dates
We use `pd.to_datetime` inside `try / except` (same idea as the `to_int` function in class): if a value can't be turned into a date
(for example `31-Apr-26`, which does not exist), we return `NaT` (an empty date).

In [7]:
def to_date(text):
    try:
        return pd.to_datetime(text)
    except Exception:
        return pd.NaT

print(to_date("25-Feb-26"), to_date("31-Apr-26"), to_date(np.nan))

2026-02-25 00:00:00 NaT NaT


### 2.3 Apply to all the date columns and build the audit log
For every value that becomes empty, we keep a row in the **audit log** with the original value and the reason.

In [8]:
def clean_dates(df):
    audit_rows = []
    for column in date_columns:
        original = df[column].copy()
        dates = pd.to_datetime(df[column].apply(clean_date_text).apply(to_date))

        # plausibility check: event dates must fall in 2025-2026, birth dates after 1900
        if column == "Date of Birth":
            out_of_range = dates.notna() & ~dates.between("1900-01-01", "2026-12-31")
        else:
            out_of_range = dates.notna() & ~dates.between("2025-01-01", "2026-12-31")
        dates[out_of_range] = pd.NaT

        df[column] = dates

        # audit log: every value that ended up empty
        empty = df[column].isna()
        audit = pd.DataFrame({"row": df.index[empty], "column": column, "original value": original[empty].values})
        audit_rows.append(audit)

    audit_log = pd.concat(audit_rows, ignore_index=True)
    return df, audit_log

def reason(value):
    if pd.isna(value):
        return "blank"
    elif any(character.isdigit() for character in str(value)):
        return "invalid or out-of-range date"
    else:
        return "text placeholder"

train, audit_train = clean_dates(train)
audit_train["reason"] = audit_train["original value"].apply(reason)
audit_train.head(10)

,row,column,original value,reason
0,1,Date of Birth,NP,text placeholder
1,4,Date of Birth,Not provided,text placeholder
2,34,Date of Birth,NaN,blank
3,35,Date of Birth,NaN,blank
4,39,Date of Birth,NaN,blank
5,43,Date of Birth,Not provided,text placeholder
6,45,Date of Birth,Not provided,text placeholder
7,54,Date of Birth,Not provided,text placeholder
8,82,Date of Birth,Not provided,text placeholder
9,96,Date of Birth,NaN,blank


### 2.4 Data-quality summary of the dates (for Q1.2)

In [9]:
n = train.shape[0]
missing_per_column = audit_train.groupby("column").size().reindex(date_columns, fill_value=0)
summary = pd.DataFrame({
    "missing / placeholder": missing_per_column,
    "% of records": (missing_per_column / n * 100).round(1),
})
display(summary)

total_cells = n * len(date_columns)
print(f"Date cells missing or placeholder: {audit_train.shape[0]} of {total_cells} "
      f"({audit_train.shape[0] / total_cells * 100:.1f}%)")
audit_train["reason"].value_counts()

,missing / placeholder,% of records
column,,
Reporting Date,0,0.0
Date of Birth,544,13.2
Date of Onset/Emergence,25,0.6
Date of Detection/Seen,11,0.3
Date Specimen Collected,2940,71.6
Date Specimen Sent,2987,72.7
Date Specimen Received at Lab,3790,92.3
Date Lab Tested/ Reviewed,3933,95.8


Date cells missing or placeholder: 14230 of 32856 (43.3%)


reason
text placeholder                13962
blank                             208
invalid or out-of-range date       60
Name: count, dtype: int64

In [10]:
# most common placeholder texts
audit_train[audit_train["reason"] == "text placeholder"]["original value"].str.strip().value_counts().head(15)

original value
No specimen collected     6475
Not applicable            1528
Pending                    950
No specimen received       932
No specimen sent           860
Not provided               672
No specimen tested         646
No Specimen collected      547
Not tested                 224
Yet to be provided          80
No specimen send            78
Not available               73
No specimen collocted.      67
Unknown                     63
Not done                    49
Name: count, dtype: int64

## 3. Logical order of the anchor dates
The standard flow is: **Onset ≤ Detection ≤ Specimen collected ≤ Sent ≤ Received at lab ≤ Lab tested**.

We compare every pair of steps and flag the records where a later step has an earlier date (e.g. specimen collected before rash onset).

In [11]:
anchor_dates = [
    "Date of Onset/Emergence",
    "Date of Detection/Seen",
    "Date Specimen Collected",
    "Date Specimen Sent",
    "Date Specimen Received at Lab",
    "Date Lab Tested/ Reviewed",
]

def check_sequence(df):
    df["date_sequence_violation"] = False
    results = []
    for i in range(len(anchor_dates)):
        for j in range(i + 1, len(anchor_dates)):
            earlier, later = anchor_dates[i], anchor_dates[j]
            wrong = df[earlier].notna() & df[later].notna() & (df[later] < df[earlier])
            if wrong.sum() > 0:
                results.append([earlier, later, wrong.sum()])
            df.loc[wrong, "date_sequence_violation"] = True
    pairs = pd.DataFrame(results, columns=["earlier step", "later step", "records"])
    return df, pairs.sort_values("records", ascending=False)

train, violations_train = check_sequence(train)
print("Records violating the date sequence:", train["date_sequence_violation"].sum(),
      f"({train['date_sequence_violation'].mean() * 100:.1f}%)")
violations_train

Records violating the date sequence: 117 (2.8%)


,earlier step,later step,records
5,Date of Detection/Seen,Date Specimen Collected,50
0,Date of Onset/Emergence,Date of Detection/Seen,41
6,Date of Detection/Seen,Date Specimen Sent,35
2,Date of Onset/Emergence,Date Specimen Sent,26
1,Date of Onset/Emergence,Date Specimen Collected,20
9,Date Specimen Collected,Date Specimen Sent,17
12,Date Specimen Sent,Date Specimen Received at Lab,7
10,Date Specimen Collected,Date Specimen Received at Lab,4
3,Date of Onset/Emergence,Date Specimen Received at Lab,4
7,Date of Detection/Seen,Date Specimen Received at Lab,4


**How we handle them:** the records are **kept** (the patient is still a real case) but flagged in
`date_sequence_violation`. In Phase 2, any time interval that comes out negative is left out of the timeliness indicators.

## 4. Patient age → Age in Years

In [12]:
train["Age Type"].value_counts(dropna=False)

Age Type
Years      3053
Months     1035
Weeks         7
Days          4
years         2
Minutes       2
days          1
NaN           1
Months        1
weeks         1
Name: count, dtype: int64

In [13]:
# fix spelling/casing/spaces: "years" -> "Years", "Months " -> "Months"
train["Age Type"] = train["Age Type"].str.strip().str.capitalize()

# "NP" is not a number -> NaN
train["Patient Age"] = pd.to_numeric(train["Patient Age"], errors="coerce")

train["Age Type"].value_counts(dropna=False)

Age Type
Years      3055
Months     1036
Weeks         8
Days          5
Minutes       2
NaN           1
Name: count, dtype: int64

In [14]:
def age_in_years(row):
    age = row["Patient Age"]
    unit = row["Age Type"]
    if unit == "Years":
        return age
    elif unit == "Months":
        return age / 12
    elif unit == "Weeks":
        return age / 52.1775
    elif unit == "Days":
        return age / 365.25
    elif unit == "Minutes":
        return age / 525960
    else:
        return np.nan

train["Age in Years"] = train.apply(age_in_years, axis=1).round(3)
train[["Patient Age", "Age Type", "Age in Years"]].sample(10, random_state=1)

,Patient Age,Age Type,Age in Years
1351,2.0,Years,2.000
1729,7.0,Years,7.000
804,2.0,Years,2.000
1548,1.0,Years,1.000
3365,6.0,Years,6.000
2136,3.0,Years,3.000
636,9.0,Months,0.750
3037,17.0,Years,17.000
718,7.0,Months,0.583
2553,13.0,Years,13.000


In [15]:
def get_age_group(age):
    if pd.isna(age):
        return np.nan
    elif age < 1:
        return "<1 year"
    elif age < 5:
        return "1-4 years"
    elif age < 10:
        return "5-9 years"
    elif age < 15:
        return "10-14 years"
    else:
        return "15+ years"

train["Age Group"] = train["Age in Years"].apply(get_age_group)
train["Age Group"].value_counts()

Age Group
1-4 years      1178
5-9 years       903
15+ years       860
<1 year         790
10-14 years     374
Name: count, dtype: int64

## 5. Vaccination status and doses

In [16]:
train["Vaccination Status"].value_counts(dropna=False)

Vaccination Status
Vaccinated by history     1440
Not vaccinated            1270
Vaccinated by card         521
Unknown                    297
Not applicable             263
Not provided               163
Vaccinated by history      119
Not Vaccinated              10
Not applicable               6
Vaccinated                   6
Not Provided                 3
Vaccinated (PEP)             2
Vaccinated                   2
Vaccinted by history         1
Vaccinated by verbatim       1
Up to date                   1
Not aplicable                1
NaN                          1
Name: count, dtype: int64

In [17]:
def vaccination_category(status):
    if pd.isna(status):
        return "Unknown"
    status = status.strip().lower()
    if status.startswith("not vac"):
        return "Not vaccinated"
    elif status.startswith("not a"):              # "Not applicable", "Not aplicable"
        return "Not applicable"
    elif status.startswith("vac") or status == "up to date":   # by card, by history, typos, (PEP)
        return "Vaccinated"
    else:                                         # "Unknown", "Not provided"
        return "Unknown"

train["Vaccination Status"] = train["Vaccination Status"].apply(vaccination_category)
train["Vaccination Status"].value_counts()

Vaccination Status
Vaccinated        2093
Not vaccinated    1280
Unknown            464
Not applicable     270
Name: count, dtype: int64

In [18]:
train["Vaccine Doses"].value_counts(dropna=False).head(25)

Vaccine Doses
1                           1343
0                           1015
Not applicable               578
Not provided                 323
Unknown                      322
2                            241
Not vaccinated                68
0.5ml                         31
Not Applicable                19
0.5ML                         17
NaN                           14
Not applicated                14
3                             12
Not applicable                11
Not Provided                   7
                   0.5ml       7
Not Vaccinated                 7
not provided                   6
Dose 1                         6
0.5 ml                         5
Vaccinated by history          5
0.5ML/1                        4
2nd                            4
1 dose                         4
not applicable                 4
Name: count, dtype: int64

In [19]:
def dose_category(row):
    dose = str(row["Vaccine Doses"]).strip().lower()
    if row["Vaccination Status"] == "Not vaccinated" or dose in ["0", "o", "not vaccinated"]:
        return "0"
    elif "2" in dose or "3" in dose:            # "2", "2nd dose", "dose 2", "3" -> 2 or more
        return "2"
    elif dose.endswith("/1"):                    # "0.5ML/1"
        return "1"
    elif "0.5" in dose or "o.5" in dose:         # 0.5 ml is the injection VOLUME, not a dose count
        return "Unknown"
    elif "1" in dose or "one" in dose:           # "1", "1 dose", "Dose 1", "1st"
        return "1"
    else:
        return "Unknown"

train["Vaccine Doses"] = train.apply(dose_category, axis=1)
train["Dose Group"] = train["Vaccine Doses"].map({"0": "Zero-Dose", "1": "1 Dose", "2": "2+ Doses", "Unknown": "Unknown"})
train["Vaccine Doses"].value_counts()

Vaccine Doses
0          1664
1          1359
Unknown     820
2           264
Name: count, dtype: int64

## 6. Other columns used in Phases 2-4

In [20]:
train["Disease/Condition"] = train["Disease/Condition"].str.strip().replace(["Yellow Fever", "MPox"], ["Yellow fever", "Mpox"])
train["Outcome"] = train["Outcome"].str.strip().str.capitalize()
train["Community Case Detection"] = train["Community Case Detection"].str.strip().str.capitalize()

def epi_classification(value):
    value = str(value).strip().lower()
    if "lab" in value or value == "confirmed":
        return "Laboratory-Confirmed"
    elif "epi" in value:
        return "Epidemiologically Linked"
    elif "clinic" in value:
        return "Clinically Compatible"
    elif "discard" in value or "not a case" in value or "negative" in value:
        return "Discarded"
    elif value in ["nan", "not provided", "not applicable"]:
        return "Missing"
    else:
        return "Suspected/Pending/Other"

def lab_result(value):
    value = str(value).strip().lower()
    if value.startswith("negative"):
        return "Negative"
    elif value.startswith("positive") or value == "confirmed":
        return "Positive"
    elif value.startswith("pending") or "sent to lab" in value:
        return "Pending"
    else:
        return "Not tested / Other"

train["Final Epi-classification (std)"] = train["Final Epi-classification"].apply(epi_classification)
train["Final Laboratory Result (std)"] = train["Final Laboratory Result"].apply(lab_result)
train = train.drop(columns=["Unnamed: 60"])

display(train["Disease/Condition"].value_counts(), train["Final Epi-classification (std)"].value_counts())

Disease/Condition
Measles                                        3295
Mpox                                            515
Adverse Event Following Immunization (AEFI)     284
Yellow fever                                      9
Acute Flaccid Paralysis (AFP)                     2
Neonatal Tetanus                                  2
Name: count, dtype: int64

Final Epi-classification (std)
Clinically Compatible       2450
Suspected/Pending/Other     1211
Epidemiologically Linked     239
Discarded                    102
Laboratory-Confirmed          67
Missing                       38
Name: count, dtype: int64

## 7. Apply exactly the same cleaning to the test data and save
We put all the steps above into one function so that train and test are cleaned the same way.

In [21]:
def clean_dataset(df):
    df, audit = clean_dates(df)
    audit["reason"] = audit["original value"].apply(reason)
    df, violations = check_sequence(df)
    df["Age Type"] = df["Age Type"].str.strip().str.capitalize()
    df["Patient Age"] = pd.to_numeric(df["Patient Age"], errors="coerce")
    df["Age in Years"] = df.apply(age_in_years, axis=1).round(3)
    df["Age Group"] = df["Age in Years"].apply(get_age_group)
    df["Vaccination Status"] = df["Vaccination Status"].apply(vaccination_category)
    df["Vaccine Doses"] = df.apply(dose_category, axis=1)
    df["Dose Group"] = df["Vaccine Doses"].map({"0": "Zero-Dose", "1": "1 Dose", "2": "2+ Doses", "Unknown": "Unknown"})
    df["Disease/Condition"] = df["Disease/Condition"].str.strip().replace(["Yellow Fever", "MPox"], ["Yellow fever", "Mpox"])
    df["Outcome"] = df["Outcome"].str.strip().str.capitalize()
    df["Community Case Detection"] = df["Community Case Detection"].str.strip().str.capitalize()
    df["Final Epi-classification (std)"] = df["Final Epi-classification"].apply(epi_classification)
    df["Final Laboratory Result (std)"] = df["Final Laboratory Result"].apply(lab_result)
    df = df.drop(columns=["Unnamed: 60"])
    return df, audit, violations

test, audit_test, violations_test = clean_dataset(pd.read_csv("test_data.csv"))
print("Test records:", test.shape[0])
print("Test date cells missing/placeholder:", audit_test.shape[0])
print("Test records violating the date sequence:", test["date_sequence_violation"].sum())

Test records: 1027
Test date cells missing/placeholder: 3544
Test records violating the date sequence: 31


In [23]:
import os
os.makedirs("outputs", exist_ok=True)

train.to_csv("outputs/cleaned_train.csv", index=False)
test.to_csv("outputs/cleaned_test.csv", index=False)
audit_train.to_csv("outputs/audit_missing_dates_train.csv", index=False)
audit_test.to_csv("outputs/audit_missing_dates_test.csv", index=False)
violations_train.to_csv("outputs/audit_date_sequence_violations_train.csv", index=False)
violations_test.to_csv("outputs/audit_date_sequence_violations_test.csv", index=False)
summary.to_csv("outputs/phase1_date_quality_summary_train.csv")

print("Saved the cleaned files and audit logs to the outputs folder")

Saved the cleaned files and audit logs to the outputs folder
